In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json, random
 
ALPHA    = 0.35
DATASET  = "triviaqa"
N_CAL    = 300   # smaller than main experiments for speed
N_TEST   = 300

In [2]:
!pip install -q sentence-transformers

In [3]:
pip install -U bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.1 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install -q gensim

In [9]:
import sys
sys.path.append('/kaggle/input/datasets/sashuvkafle/efficient')

In [11]:
from model import load_sentence_encoder, load_model, generate_response, generate_batch
from scoring import clean_response, compute_stability, is_uncertainty_response, plateau_stop, responses_are_substantive
from data import load_triviaqa, load_mmlu, load_webquestions
from evaluate import evaluate, run_full_evaluation, print_comparison_table
from calibration import compute_qhat, check_coverage, run_calibration
from sampler import adaptive_sample, check_coverage, build_prediction_set
from helpers import _assign_split, _print_split_counts, filter_split, clear_embed_cache, embed_cached
from scoring import clean_response, compute_stability, plateau_stop, is_uncertainty_response, responses_are_substantive, token_f1
from config import QASample, SamplerConfig, UNCERTAINTY_TOKENS, COVERAGE_SIM_THRESHOLD, MMLU_SUBJECTS

Loading mistralai/Mistral-7B-Instruct-v0.3 ...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded.
GPU memory used: 1.6 GB


In [14]:
DEFAULT_CONFIG = SamplerConfig()

In [16]:
def evaluate_ablation(
    dataset_name:   str,
    alpha:          float,
    sampler_fn,                          # which adaptive_sample variant to use
    cal_samples:    int   = 500,
    test_samples:   int   = 500,
    config:         SamplerConfig = DEFAULT_CONFIG,
) -> dict:
    """
    Calibrate using sampler_fn on calibration split,
    evaluate on test split. Returns ECR, SSC, APSS, API, abstain_rate.
    """
    import random

    # --- Calibration ---
    cal = filter_split(DATASETS[dataset_name], "calibration")
    if len(cal) > cal_samples:
        cal = random.sample(cal, cal_samples)

    n_scores    = []
    n_abstained = 0

    for sample in cal:
        clear_embed_cache()
        result = sampler_fn(sample.question, config=config)
        if result["abstain"]:
            n_abstained += 1
        else:
            n_scores.append(result["n_score"])

    q_hat = compute_qhat(n_scores, alpha)
    cal_abstain_rate = n_abstained / len(cal)

    print(f"  Calibration: n={len(cal)}, n_scored={len(n_scores)}, "
          f"n_abstained={n_abstained}, q_hat={q_hat:.4f}")

    # --- Evaluation ---
    test = filter_split(DATASETS[dataset_name], "test")
    if len(test) > test_samples:
        test = random.sample(test, test_samples)

    n_covered     = 0
    n_ssc_covered = 0
    n_ssc_total   = 0
    total_set_size  = 0
    total_api_calls = 0

    for i, sample in enumerate(test):
        clear_embed_cache()
        result   = sampler_fn(sample.question, config=config)
        pred_set = build_prediction_set(result, q_hat)
        covered  = check_coverage(pred_set, sample.gold_answers)
        abstained = result["abstain"] or (
            result["n_score"] is not None and result["n_score"] > q_hat
        )

        if covered:
            n_covered += 1
        if not abstained:
            n_ssc_total += 1
            if covered:
                n_ssc_covered += 1

        total_set_size  += len(pred_set)
        total_api_calls += result["n_samples"]

        if (i + 1) % 100 == 0:
            print(f"  test [{i+1}/{len(test)}] ECR={n_covered/(i+1)*100:.1f}%")

    n = len(test)
    return {
        "ECR":          round(n_covered / n * 100, 1),
        "SSC":          round((n_ssc_covered / n_ssc_total * 100)
                              if n_ssc_total > 0 else 0.0, 1),
        "APSS":         round(total_set_size / n, 2),
        "API":          round(total_api_calls / n, 1),
        "abstain_rate": round((n - n_ssc_total) / n * 100, 1),
        "q_hat":        q_hat,
    }

In [21]:
import os
import json
import random
import numpy as np
from collections import Counter
from typing import Optional
import torch
from datasets import load_dataset
import random
import time
import math

from sentence_transformers import SentenceTransformer

from dataclasses import dataclass, field

In [22]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [23]:
triviaqa_samples   = load_triviaqa(SEED, n_total=3000)
webq_samples       = load_webquestions(SEED)
mmlu_samples       = load_mmlu(SEED)

DATASETS = {
    "triviaqa": triviaqa_samples,
    "webq":     webq_samples,
    "mmlu":     mmlu_samples,
}

# Quick sanity check
for name, samples in DATASETS.items():
    cal  = len(filter_split(samples, "calibration"))
    val  = len(filter_split(samples, "validation"))
    test = len(filter_split(samples, "test"))
    print(f"{name}: cal={cal}, val={val}, test={test}")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

rc.nocontext/train-00000-of-00001.parque(…):   0%|          | 0.00/55.4M [00:00<?, ?B/s]

rc.nocontext/validation-00000-of-00001.p(…):   0%|          | 0.00/7.34M [00:00<?, ?B/s]

rc.nocontext/test-00000-of-00001.parquet:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

...

abstract_algebra/dev-00000-of-00001.parq(…):   0%|          | 0.00/3.45k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

MMLU loaded: 2314 samples across 5 subjects
  calibration: 1157
  validation: 579
  test: 578
triviaqa: cal=1500, val=750, test=750
webq: cal=1889, val=945, test=944
mmlu: cal=1157, val=579, test=578


In [24]:
EPS_VALUES = [0.01, 0.02, 0.03, 0.05, 0.08, 0.10]
 
def make_eps_config(eps):
    return SamplerConfig(
        batch_size=3, max_batches=6, eps=eps,
        temperature=0.9, min_batches=2
    )
 
print("Running epsilon sweep...")
eps_results = {}
 
for eps in EPS_VALUES:
    print(f"\n  ε={eps}")
    r = evaluate_ablation(
        dataset_name = DATASET,
        alpha        = ALPHA,
        sampler_fn   = lambda q, config=make_eps_config(eps): adaptive_sample(q, config),
        cal_samples  = N_CAL,
        test_samples = N_TEST,
    )
    eps_results[eps] = r
    print(f"  → ECR={r['ECR']}  SSC={r['SSC']}  APSS={r['APSS']:.2f}  "
          f"API={r['API']:.1f}  Abstain={r['abstain_rate']}%")

Running epsilon sweep...

  ε=0.01
Loading sentence encoder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoder loaded. Embedding dim: 384
  Calibration: n=300, n_scored=229, n_abstained=71, q_hat=-0.0000
  test [100/300] ECR=94.0%
  test [200/300] ECR=91.5%
  test [300/300] ECR=91.3%
  → ECR=91.3  SSC=83.1  APSS=0.51  API=11.9  Abstain=48.7%

  ε=0.02
  Calibration: n=300, n_scored=233, n_abstained=67, q_hat=0.0169
  test [100/300] ECR=86.0%
  test [200/300] ECR=90.5%
  test [300/300] ECR=91.3%
  → ECR=91.3  SSC=83.2  APSS=0.54  API=12.1  Abstain=48.3%

  ε=0.03
  Calibration: n=300, n_scored=232, n_abstained=68, q_hat=-0.0000
  test [100/300] ECR=86.0%
  test [200/300] ECR=90.0%
  test [300/300] ECR=90.0%
  → ECR=90.0  SSC=81.2  APSS=0.54  API=11.9  Abstain=46.7%

  ε=0.05
  Calibration: n=300, n_scored=229, n_abstained=71, q_hat=-0.0000
  test [100/300] ECR=89.0%
  test [200/300] ECR=91.5%
  test [300/300] ECR=91.7%
  → ECR=91.7  SSC=84.1  APSS=0.53  API=12.1  Abstain=47.7%

  ε=0.08
  Calibration: n=300, n_scored=219, n_abstained=81, q_hat=0.0000
  test [100/300] ECR=96.0%
  test [20

In [ ]:
# -----------------------------------------------------------------------------
# CELL S2 — Batch size sweep
# Values: 1, 2, 3, 5, 6
# 3 is our default. Batch size 1 = one sample at a time (most granular).
# Batch size 6 = only 3 batches before max budget.
# Keep max_batches*batch_size = 18 (fixed total budget) for fair comparison.
# -----------------------------------------------------------------------------
 
# batch_size × max_batches = 18 always (fixed total budget)
BATCH_CONFIGS = [
    (1, 18),   # 18 batches of 1 — very granular stopping
    (2,  9),   # 9 batches of 2
    (3,  6),   # 6 batches of 3 — our default
    (6,  3),   # 3 batches of 6 — coarse stopping
    (9,  2),   # 2 batches of 9 — almost no adaptivity
]
 
def make_batch_config(batch_size, max_batches):
    return SamplerConfig(
        batch_size=batch_size, max_batches=max_batches,
        eps=0.03, temperature=0.9,
        min_batches=min(2, max_batches),
    )
 
print("\n\nRunning batch size sweep...")
batch_results = {}
 
for bs, mb in BATCH_CONFIGS:
    print(f"\n  batch_size={bs}, max_batches={mb} (budget={bs*mb})")
    cfg = make_batch_config(bs, mb)
    r = evaluate_ablation(
        dataset_name = DATASET,
        alpha        = ALPHA,
        sampler_fn   = lambda q, config=cfg: adaptive_sample(q, config),
        cal_samples  = N_CAL,
        test_samples = N_TEST,
    )
    batch_results[bs] = r
    print(f"  → ECR={r['ECR']}  SSC={r['SSC']}  APSS={r['APSS']:.2f}  "
          f"API={r['API']:.1f}  Abstain={r['abstain_rate']}%")

In [ ]:
plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        11,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "grid.linestyle":   "--",
    "figure.dpi":       150,
})
 
DEFAULT_COLOR  = "#2563EB"   # blue  — default config
OTHER_COLOR    = "#94A3B8"   # slate — other configs
HIGHLIGHT      = "#DC2626"   # red   — worst performer
TARGET_COLOR   = "#16A34A"   # green — ECR target line
 
DEFAULT_EPS   = 0.03
DEFAULT_BATCH = 3
 
 
def make_sensitivity_figure(param_values, results_dict, default_val,
                             xlabel, title_suffix, filename,
                             x_is_log=False):
    """
    Generic 2×2 sensitivity plot for ECR, SSC, APSS, API.
    Highlights the default config in blue, others in grey.
    """
    metrics = [
        ("ECR",          "ECR (%)",            True),   # (key, ylabel, draw_target)
        ("SSC",          "SSC (%)",            False),
        ("APSS",         "APSS",               False),
        ("API",          "Avg API Calls",      False),
    ]
 
    xs    = param_values
    x_num = list(range(len(xs)))
 
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    fig.suptitle(f"Sensitivity to {title_suffix}  |  TriviaQA, α=0.35",
                 fontsize=14, fontweight="bold", y=1.01)
 
    for ax, (key, ylabel, draw_target) in zip(axes.flat, metrics):
        ys = [results_dict[x][key] for x in xs]
 
        # Bar colors
        colors = [DEFAULT_COLOR if x == default_val else OTHER_COLOR for x in xs]
 
        bars = ax.bar(x_num, ys, color=colors, width=0.55, zorder=3,
                      edgecolor="white", linewidth=0.8)
 
        # Value labels on bars
        for bar, y in zip(bars, ys):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(ys) * 0.01,
                    f"{y:.1f}", ha="center", va="bottom",
                    fontsize=9, color="#1E293B")
 
        # ECR target line
        if draw_target:
            target = (1 - ALPHA) * 100
            ax.axhline(target, color=TARGET_COLOR, linewidth=1.5,
                       linestyle="--", zorder=2, label=f"Target ({target:.0f}%)")
            ax.legend(fontsize=9, framealpha=0.7)
 
        ax.set_xticks(x_num)
        ax.set_xticklabels([str(x) for x in xs], fontsize=10)
        ax.set_xlabel(xlabel, fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
 
        # Shade the default bar subtly
        default_idx = xs.index(default_val)
        ax.get_xticklabels()[default_idx].set_fontweight("bold")
        ax.get_xticklabels()[default_idx].set_color(DEFAULT_COLOR)
 
        # Y axis padding
        ymin = min(ys) * 0.92
        ymax = max(ys) * 1.08
        ax.set_ylim(ymin, ymax)
 
        # Subtitle showing range
        spread = max(ys) - min(ys)
        ax.set_title(f"{ylabel}  (range: {spread:.2f})", fontsize=10,
                     color="#475569", pad=6)
 
    # Legend for colors
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=DEFAULT_COLOR, label="Default config"),
        Patch(facecolor=OTHER_COLOR,   label="Other configs"),
    ]
    fig.legend(handles=legend_elements, loc="lower center",
               ncol=2, fontsize=10, framealpha=0.9,
               bbox_to_anchor=(0.5, -0.04))
 
    plt.tight_layout()
    plt.savefig(f"/content/{filename}", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: /content/{filename}")
 
 
# Figure 1: Epsilon sensitivity
make_sensitivity_figure(
    param_values = EPS_VALUES,
    results_dict = eps_results,
    default_val  = DEFAULT_EPS,
    xlabel       = "ε (convergence tolerance)",
    title_suffix = "ε (Plateau Stopping Threshold)",
    filename     = "sensitivity_epsilon.png",
)
 
# Figure 2: Batch size sensitivity
make_sensitivity_figure(
    param_values = [bs for bs, _ in BATCH_CONFIGS],
    results_dict = batch_results,
    default_val  = DEFAULT_BATCH,
    xlabel       = "Batch size (total budget fixed at 18)",
    title_suffix = "Batch Size",
    filename     = "sensitivity_batchsize.png",
)

In [ ]:
fig = plt.figure(figsize=(14, 9))
fig.patch.set_facecolor("#F8FAFC")
 
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.38)
 
metrics_cfg = [
    ("ECR",  "ECR (%)",       "#2563EB", True),
    ("SSC",  "SSC (%)",       "#7C3AED", False),
    ("APSS", "APSS",          "#D97706", False),
    ("API",  "Avg API Calls", "#059669", False),
]
 
# Row 0: epsilon sweep (line plot)
for col, (key, ylabel, color, draw_target) in enumerate(metrics_cfg):
    ax = fig.add_subplot(gs[0, col])
    xs = EPS_VALUES
    ys = [eps_results[x][key] for x in xs]
 
    ax.plot(xs, ys, color=color, linewidth=2.5, marker="o",
            markersize=7, markerfacecolor="white", markeredgewidth=2.5,
            zorder=3)
 
    # Highlight default
    default_y = eps_results[DEFAULT_EPS][key]
    ax.scatter([DEFAULT_EPS], [default_y], s=120, color=color,
               zorder=4, label="Default (ε=0.03)")
 
    if draw_target:
        ax.axhline((1-ALPHA)*100, color="#DC2626", linewidth=1.2,
                   linestyle="--", alpha=0.7, label="Target")
 
    ax.set_xlabel("ε", fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(ylabel, fontsize=11, fontweight="bold", color="#1E293B")
 
    if col == 0:
        ax.set_ylabel("ε Sweep\n" + ylabel, fontsize=10)
 
    spread = max(ys) - min(ys)
    ax.set_ylim(min(ys) * 0.93, max(ys) * 1.07)
 
    # Annotate spread
    ax.text(0.97, 0.05, f"Δ={spread:.2f}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=9, color="#64748B",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                      edgecolor="#CBD5E1", alpha=0.8))
 
# Row 1: batch size sweep (line plot)
batch_xs = [bs for bs, _ in BATCH_CONFIGS]
for col, (key, ylabel, color, draw_target) in enumerate(metrics_cfg):
    ax = fig.add_subplot(gs[1, col])
    ys = [batch_results[bs][key] for bs in batch_xs]
 
    ax.plot(batch_xs, ys, color=color, linewidth=2.5, marker="s",
            markersize=7, markerfacecolor="white", markeredgewidth=2.5,
            zorder=3)
 
    default_y = batch_results[DEFAULT_BATCH][key]
    ax.scatter([DEFAULT_BATCH], [default_y], s=120, color=color,
               zorder=4, label=f"Default (b=3)")
 
    if draw_target:
        ax.axhline((1-ALPHA)*100, color="#DC2626", linewidth=1.2,
                   linestyle="--", alpha=0.7, label="Target")
 
    ax.set_xlabel("Batch size", fontsize=10)
    ax.set_xticks(batch_xs)
 
    if col == 0:
        ax.set_ylabel("Batch Size Sweep\n" + ylabel, fontsize=10)
    else:
        ax.set_ylabel(ylabel, fontsize=10)
 
    spread = max(ys) - min(ys)
    ax.set_ylim(min(ys) * 0.93, max(ys) * 1.07)
 
    ax.text(0.97, 0.05, f"Δ={spread:.2f}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=9, color="#64748B",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                      edgecolor="#CBD5E1", alpha=0.8))
 
fig.suptitle(
    "Hyperparameter Sensitivity  |  TriviaQA, α=0.35\n"
    "Filled markers = default configuration",
    fontsize=13, fontweight="bold", y=1.01, color="#0F172A"
)
 
plt.savefig("/content/sensitivity_combined.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("Saved: /content/sensitivity_combined.png")
 

In [ ]:
print("\n" + "="*60)
print("EPSILON SENSITIVITY  (TriviaQA, alpha=0.35)")
print("="*60)
print(f"  {'ε':<8} {'ECR':>6} {'SSC':>6} {'APSS':>6} {'API':>5} {'Abstain':>8}")
print(f"  {'-'*46}")
for eps in EPS_VALUES:
    r = eps_results[eps]
    marker = " ← default" if eps == DEFAULT_EPS else ""
    ok = "✓" if r["ECR"] >= (1-ALPHA)*100 else "✗"
    print(f"  {eps:<8} {r['ECR']:>5.1f}{ok} {r['SSC']:>6.1f} "
          f"{r['APSS']:>6.2f} {r['API']:>5.1f} {r['abstain_rate']:>7.1f}%{marker}")
 
ecr_vals  = [eps_results[e]["ECR"]  for e in EPS_VALUES]
ssc_vals  = [eps_results[e]["SSC"]  for e in EPS_VALUES]
apss_vals = [eps_results[e]["APSS"] for e in EPS_VALUES]
print(f"\n  ECR  range: {min(ecr_vals):.1f}–{max(ecr_vals):.1f}  "
      f"(spread={max(ecr_vals)-min(ecr_vals):.1f}pp)")
print(f"  SSC  range: {min(ssc_vals):.1f}–{max(ssc_vals):.1f}  "
      f"(spread={max(ssc_vals)-min(ssc_vals):.1f}pp)")
print(f"  APSS range: {min(apss_vals):.2f}–{max(apss_vals):.2f}  "
      f"(spread={max(apss_vals)-min(apss_vals):.2f})")
 
print("\n" + "="*60)
print("BATCH SIZE SENSITIVITY  (TriviaQA, alpha=0.35, budget=18)")
print("="*60)
print(f"  {'batch':>5} {'ECR':>6} {'SSC':>6} {'APSS':>6} {'API':>5} {'Abstain':>8}")
print(f"  {'-'*46}")
for bs, mb in BATCH_CONFIGS:
    r = batch_results[bs]
    marker = " ← default" if bs == DEFAULT_BATCH else ""
    ok = "✓" if r["ECR"] >= (1-ALPHA)*100 else "✗"
    print(f"  {bs:>5} {r['ECR']:>5.1f}{ok} {r['SSC']:>6.1f} "
          f"{r['APSS']:>6.2f} {r['API']:>5.1f} {r['abstain_rate']:>7.1f}%{marker}")
 
ecr_vals  = [batch_results[bs]["ECR"]  for bs, _ in BATCH_CONFIGS]
ssc_vals  = [batch_results[bs]["SSC"]  for bs, _ in BATCH_CONFIGS]
print(f"\n  ECR  range: {min(ecr_vals):.1f}–{max(ecr_vals):.1f}  "
      f"(spread={max(ecr_vals)-min(ecr_vals):.1f}pp)")
print(f"  SSC  range: {min(ssc_vals):.1f}–{max(ssc_vals):.1f}  "
      f"(spread={max(ssc_vals)-min(ssc_vals):.1f}pp)")
 
# Save results
with open("/content/sensitivity_results.json", "w") as f:
    json.dump({
        "epsilon":    {str(k): v for k, v in eps_results.items()},
        "batch_size": {str(k): v for k, v in batch_results.items()},
    }, f, indent=2)
print("\nSaved: /content/sensitivity_results.json")
